In [5]:
# ================= MULTI-DATE ORCHESTRATOR (APPEND RESULTS) =================
import os, math
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ILP solver
try:
    import pulp as pl
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pulp", "-q"])
    import pulp as pl

# Paths
INPUTDIR = "."  # folder containing raw_init_YYYY-MM-DD.csv
FILE_HOSPITAL = "./hospital_model_inputs.xlsx"
FILE_COMPAT = "./compatibility_table.xlsx"
OUTDIR = "./outputs"
os.makedirs(OUTDIR, exist_ok=True)

# Configure the exact dates you want to solve (independently, one-day each) 
#******************************CONFIGURATION******************************#
SOLVE_DATES = [
    "2025-07-06",
    "2025-07-07",
    "2025-07-08",
    "2025-07-09",
    "2025-07-10",
    "2025-07-11",
    "2025-07-12",
]

# Where to find each day's midnight census CSV (one file per date)
# Example: raw_init_2025-07-06.csv, raw_init_2025-07-07.csv, ...
RAW_INIT_TMPL = os.path.join(INPUTDIR, "raw_init_{date}.csv")  # edit if needed

# Combined output Excel (single file for all dates)
_combined_name = f"optimized_plan_{SOLVE_DATES[0]}_to_{SOLVE_DATES[-1]}.xlsx"
COMBINED_XLSX = os.path.join(OUTDIR, _combined_name)

# ------------- helpers: loaders (reuse your existing ones if you have them) -------------
def load_hospital_inputs_once(xlsx_path):
    beds_df = pd.read_excel(xlsx_path, sheet_name="beds")
    beds_df.columns = [c.strip() for c in beds_df.columns]
    assert {"Unit","Beds"}.issubset(beds_df.columns), "Sheet 'beds' must have Unit, Beds"
    beds_df["Unit"] = beds_df["Unit"].astype(str).str.strip()
    beds_df["Beds"] = pd.to_numeric(beds_df["Beds"], errors="coerce").fillna(0).astype(int)

    fc_df = pd.read_excel(xlsx_path, sheet_name="forecasts")
    fc_df.columns = [c.strip() for c in fc_df.columns]
    assert {"Unit","Admit_Avg","Disch_Avg"}.issubset(fc_df.columns), "Sheet 'forecasts' must have Unit, Admit_Avg, Disch_Avg"
    fc_df["Unit"] = fc_df["Unit"].astype(str).str.strip()
    fc_df["Admit_Avg"] = pd.to_numeric(fc_df["Admit_Avg"], errors="coerce").fillna(0)
    fc_df["Disch_Avg"] = pd.to_numeric(fc_df["Disch_Avg"], errors="coerce").fillna(0)

    return beds_df, fc_df

def load_raw_init_for_date(csv_path):
    raw_df = pd.read_csv(csv_path)
    raw_df.columns = [c.strip() for c in raw_df.columns]
    if not {"Unit","Raw_Census"}.issubset(raw_df.columns):
        raise ValueError(f"{csv_path} must include columns: Unit, Raw_Census")
    raw_df["Unit"] = raw_df["Unit"].astype(str).str.strip()
    raw_df["Raw_Census"] = pd.to_numeric(raw_df["Raw_Census"], errors="coerce").fillna(0).astype(int)
    return raw_df

def load_compat(path_like):
    """
    Read the 3-column sheet 'compatibility_table' from `path_like`.
    Requires EXACT columns:
        'origin', 'destination', 'count'
    Returns:
      units:   list[str]  (union of origins and destinations)
      allowed: list[(origin, destination)] where count>0 and origin!=destination
    """
    import os
    import pandas as pd

    if not os.path.exists(path_like):
        raise FileNotFoundError(f"Compatibility file not found: {path_like}")

    # Ensure the sheet exists (fail fast if not)
    xls = pd.ExcelFile(path_like, engine="openpyxl")
    if "compatibility_table" not in xls.sheet_names:
        raise ValueError(
            f"'compatibility_table' sheet not found in {path_like}. "
            f"Sheets present: {xls.sheet_names}"
        )

    # Read exactly that sheet and require exact headers
    df = pd.read_excel(path_like, sheet_name="compatibility_table", engine="openpyxl")

    required = ["origin", "destination", "count"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns in 'compatibility_table': {missing}. "
            f"Found columns: {list(df.columns)}"
        )

    # Keep exact names; only coerce count
    df = df[required].copy()
    df["count"] = pd.to_numeric(df["count"], errors="coerce").fillna(0).astype(int)

    # Allowed arcs = count>0 and no self-loops
    allowed_df = df[(df["count"] > 0) & (df["origin"] != df["destination"])]

    # Build outputs (names preserved)
    units = sorted(
        set(allowed_df["origin"].astype(str)).union(
            set(allowed_df["destination"].astype(str))
        )
    )
    allowed = list(
        allowed_df[["origin", "destination"]].itertuples(index=False, name=None)
    )

    # Safety: ensure there is at least one arc
    if not allowed:
        raise ValueError("No allowed arcs found (all counts are 0 or only self-loops).")

    return units, allowed



# ------------- core: build+solve a single day and collect the 4-tab payload -------------
def solve_one_day_and_collect(day_str, file_raw_init,
                              beds_df, fc_df, compat_units, ALLOWED):
    # Merge inputs for this day
    df = (beds_df.merge(load_raw_init_for_date(file_raw_init), on="Unit", how="left")
                  .merge(fc_df, on="Unit", how="left"))
    if df["Raw_Census"].isna().any():
        missing = df[df["Raw_Census"].isna()]["Unit"].tolist()
        raise ValueError(f"Units missing Raw_Census in {os.path.basename(file_raw_init)}: {missing}")

    df["A"] = df["Admit_Avg"].fillna(0).round().astype(int)
    df["D"] = df["Disch_Avg"].fillna(0).round().astype(int)

    # dictionaries
    locations = df["Unit"].tolist()
    beds_dict = dict(zip(df["Unit"], df["Beds"]))              # Beds = STAFFED capacity
    C0_dict   = dict(zip(df["Unit"], df["Raw_Census"]))
    A_dict    = dict(zip(df["Unit"], df["A"]))
    D_dict    = dict(zip(df["Unit"], df["D"]))

    # ---- BUILD ILP ----
    model = pl.LpProblem(f"OneDay_{day_str}", pl.LpMinimize)

    Occ           = pl.LpVariable.dicts("Occ",           [(u,1) for u in locations], lowBound=0, cat=pl.LpInteger)
    In85          = pl.LpVariable.dicts("In85",          [(u,1) for u in locations], lowBound=0, cat=pl.LpInteger)
    Ov85          = pl.LpVariable.dicts("Ov85",          [(u,1) for u in locations], lowBound=0, cat=pl.LpInteger)
    StayOverflow  = pl.LpVariable.dicts("StayOverflow",  [(u,1) for u in locations], lowBound=0, cat=pl.LpInteger)
    Trans         = pl.LpVariable.dicts("Trans",         [(i,j,1) for (i,j) in ALLOWED], lowBound=0, cat=pl.LpInteger)
    Near95        = pl.LpVariable.dicts("Near95",        [(u,1) for u in locations],lowBound=0, cat=pl.LpContinuous)


    # Mass balance with transfers
    for u in locations:
        inflow  = pl.lpSum(Trans[(i,u,1)] for (i,j) in ALLOWED if j==u)
        outflow = pl.lpSum(Trans[(u,j,1)] for (i,j) in ALLOWED if i==u)
        model += ( Occ[(u,1)] == C0_dict[u] - D_dict[u] + A_dict[u] + inflow - outflow, f"MassBal_{u}" )

    # Band decomposition + 85/95 managerial thresholds on Beds (staffed)
    for u in locations:
        b = int(beds_dict[u])
        model += ( Occ[(u,1)] == In85[(u,1)] + Ov85[(u,1)] + StayOverflow[(u,1)], f"Decomp_{u}" )
        model += ( In85[(u,1)] <= math.floor(0.85*b),                     f"In85Cap_{u}" )
        model += (In85[(u,1)] + Ov85[(u,1)] <= b, f"InBedsCap_{u}")
        cap95 = math.floor(0.95 * b)
        model += (Near95[(u,1)] >= In85[(u,1)] + Ov85[(u,1)] - cap95, f"Near95Def_{u}")

    # Objective: strongly penalize overflow; lightly penalize transfers
    W_OVERFLOW = 1000.0   # dominant
    W_TRANSFER = 1.0      # secondary
    W_NEAR95   = 0.1      # small soft penalty for using the last 5% of beds
    
    # Preference for staying ≤85% (Ov85 = patients in 85–95% band)
    ENFORCE_COMFORT = False          # set False to disable the nudge
    W_OV85 = 0 if ENFORCE_COMFORT else 0.0   # tie-breaker;  set to 0 for now 

    model += (
        W_OVERFLOW * pl.lpSum(StayOverflow[(u,1)] for u in locations) +
        W_TRANSFER * pl.lpSum(Trans[(i,j,1)]      for (i,j) in ALLOWED) +
        W_NEAR95   * pl.lpSum(Near95[(u,1)]       for u in locations)  
)


    model.solve(pl.PULP_CBC_CMD(msg=False))

# ---- RESULTS COLLECT (build exactly your 4 tabs for this day) ----
    def val(x): 
        v = pl.value(x)
        return int(round(v if v is not None else 0))

    beds_s  = pd.Series({u: beds_dict[u] for u in locations}, name="Beds").astype(int)
    in85_s  = pd.Series({u: val(In85[(u,1)])         for u in locations})
    ov85_s  = pd.Series({u: val(Ov85[(u,1)])         for u in locations})
    stay_s  = pd.Series({u: val(StayOverflow[(u,1)]) for u in locations})
    occ_s   = pd.Series({u: val(Occ[(u,1)])          for u in locations})
    val = pl.value

    in85_s   = pd.Series({u: val(In85[(u,1)])         for u in locations}, name="In85_InBeds")
    ov85_s   = pd.Series({u: val(Ov85[(u,1)])         for u in locations}, name="Ov85_InBeds")
    stay_s   = pd.Series({u: val(StayOverflow[(u,1)]) for u in locations}, name="StayOverflow")
    occ_s    = pd.Series({u: val(Occ[(u,1)])          for u in locations}, name="TrueCensus")
    near95_s = pd.Series({u: val(Near95[(u,1)])       for u in locations}, name="PtLast5%")
    beds_s   = pd.Series({u: int(beds_dict[u])        for u in locations}, name="Beds").astype(float)

# Derived
    staffed_inbeds = (in85_s + ov85_s).rename("StaffedInBeds")
    true_census    = occ_s.rename("TrueCensus")
    true_pct       = (100.0 * true_census / beds_s)
    

    staffed_used = (in85_s + ov85_s)  # beds_used (no overflows)
    occ_pct      = (100.0 * staffed_used / beds_s.replace(0, np.nan)).fillna(0.0)
    true_pct     = (100.0 * occ_s       / beds_s.replace(0, np.nan)).fillna(0.0)

    # Tab 1: Optimization bedroster
    occ_tab = pd.DataFrame({
        "date": day_str,
        "unit": locations,
        "beds_used": staffed_used.astype(int).values,
        "beds": beds_s.astype(int).values,
        "Optimized Bedroster%": true_pct.round(1).values,
        "PtLast5%": near95_s.round().astype(int).values,
        "Overflow":  stay_s.round().astype(int).reindex(locations).values,
    })

    #Tab 2: Transfers
    # Define start-of-day backlog as excess over staffed beds
    backlog0 = {u: max(0, int(C0_dict[u]) - int(beds_dict[u])) for u in locations}

    rows = []
    for u in locations:
    # outgoing arcs and solved counts
        out_arcs = [(u,v) for (i,v) in ALLOWED if i == u]
        tcounts  = {v: int(round(val(Trans[(u,v,1)]))) for v in [j for (_,j) in out_arcs]}
        tot_out  = sum(tcounts.values())
        if tot_out == 0:
            continue

    # how many of u's outflows can be labeled backlog today
        bl_rem = min(backlog0.get(u, 0), tot_out)

    # proportional allocation of backlog across outgoing arcs (with rounding)
        if bl_rem > 0:
            shares = {v: (tcounts[v] / tot_out) for v in tcounts}
            alloc  = {v: int(shares[v] * bl_rem) for v in tcounts}
            rem    = bl_rem - sum(alloc.values())
            if rem > 0:
                fracs = sorted([(shares[v]*bl_rem - alloc[v], v) for v in tcounts], reverse=True)
                for k in range(rem):
                    alloc[fracs[k][1]] += 1
        else:
            alloc = {v: 0 for v in tcounts}

    # emit rows: first 'backlog' (if any), then 'new transfer' remainder
        for v in tcounts:
            tij = tcounts[v]
            bl  = min(alloc[v], tij)
            nw  = tij - bl
            if bl > 0:
                rows.append([day_str, u, v, bl, "backlog"])
            if nw > 0:
                rows.append([day_str, u, v, nw, "new transfer"])

    transfers_tab = pd.DataFrame(rows, columns=[
        "date", "origin unit", "destination unit", "patient (number)", "type (new transfer or backlog)"
])

    # Tab 3: Overflows (your policy: no unplaced)
    of_rows = [[day_str, u, int(stay_s[u])] for u in locations if int(stay_s[u])>0]
    overflows_tab = pd.DataFrame(of_rows, columns=["date","unit","overflows number"])

    # Heatmap contributions (per-date columns)
    staffed_pct_col = pd.Series(occ_pct.round(1).values, index=locations, name=day_str)
    true_pct_col    = pd.Series(true_pct.round(1).values, index=locations, name=day_str)

    return {
        "occ_tab": occ_tab,
        "transfers_tab": transfers_tab,
        "overflows_tab": overflows_tab,
        "staffed_pct_col": staffed_pct_col,
        "true_pct_col": true_pct_col,
        "units": locations,
    }

# ------------- run the list of dates (independent solves), then write one Excel -------------
beds_df, fc_df = load_hospital_inputs_once(FILE_HOSPITAL)
compat_units, ALLOWED = load_compat(FILE_COMPAT)  # compatibility_matrix.xlsx

all_occ_rows = []
all_transfers_rows = []
all_overflows_rows = []
# collect matrices across dates
staffed_cols = []
true_cols = []
all_units = set()

for d in SOLVE_DATES:
    file_raw = RAW_INIT_TMPL.format(date=d)
    res = solve_one_day_and_collect(d, file_raw, beds_df, fc_df, compat_units, ALLOWED)

    all_occ_rows.append(res["occ_tab"])
    if not res["transfers_tab"].empty:
        all_transfers_rows.append(res["transfers_tab"])
    if not res["overflows_tab"].empty:
        all_overflows_rows.append(res["overflows_tab"])

    staffed_cols.append(res["staffed_pct_col"])
    true_cols.append(res["true_pct_col"])
    all_units.update(res["units"])

# Build heatmap matrices (Units x Dates)
units_order = sorted(all_units)  # or keep a custom order if you prefer
staffed_matrix = pd.concat(staffed_cols, axis=1).reindex(units_order)
true_matrix    = pd.concat(true_cols,    axis=1).reindex(units_order)

#NEW 
# ---------- Concat rows & build transfer annotations for main tab ----------
occ_tab_all = pd.concat(all_occ_rows, ignore_index=True)

if len(all_transfers_rows):
    transfers_tab_all = pd.concat(all_transfers_rows, ignore_index=True).copy()

    # Ensure proper dtypes
    transfers_tab_all["patient (number)"] = (
        pd.to_numeric(transfers_tab_all["patient (number)"], errors="coerce")
        .fillna(0).astype(int)
    )
    transfers_tab_all["destination unit"] = (
        transfers_tab_all["destination unit"].astype(str).str.strip()
    )
    transfers_tab_all["type (new transfer or backlog)"] = (
        transfers_tab_all["type (new transfer or backlog)"].astype(str).str.strip()
    )

    # Aggregate per (date, origin) WITHOUT embedding counts in the destination column.
    # We keep duplicates (same destination appearing multiple rows) and build a parallel
    # "(1, 2, ...)" list for 'number of transfer' in the same order.
    def _agg_group(g):
        dests = g["destination unit"].tolist()               # e.g., ["SOINS...", "SOINS..."]
        nums  = g["patient (number)"].tolist()               # e.g., [1, 1]
        types = sorted({t for t in g["type (new transfer or backlog)"] if pd.notna(t)})
        return pd.Series({
            "destination unit": " ; ".join(dests) if dests else None,
            "number of transfer": f"({', '.join(str(int(n)) for n in nums)})" if nums else None,
            "type of transfer": ", ".join(types) if types else None
        })

# Build compact transfers without groupby.apply (no deprecation warning)
if len(all_transfers_rows):
    transfers_tab_all = pd.concat(all_transfers_rows, ignore_index=True).copy()

    # Clean dtypes
    transfers_tab_all["patient (number)"] = (
        pd.to_numeric(transfers_tab_all["patient (number)"], errors="coerce")
        .fillna(0).astype(int)
    )
    transfers_tab_all["destination unit"] = transfers_tab_all["destination unit"].astype(str).str.strip()
    transfers_tab_all["type (new transfer or backlog)"] = transfers_tab_all[
        "type (new transfer or backlog)"
    ].astype(str).str.strip()

import numpy as np

transfers_tab_all = pd.concat(all_transfers_rows, ignore_index=True).copy()
transfers_tab_all["patient (number)"] = pd.to_numeric(
    transfers_tab_all["patient (number)"], errors="coerce"
).fillna(0).astype(int)
transfers_tab_all["destination unit"] = transfers_tab_all["destination unit"].astype(str).str.strip()
transfers_tab_all["type (new transfer or backlog)"] = transfers_tab_all[
    "type (new transfer or backlog)"
].astype(str).str.strip()

# One row per (date, origin); total only
transfers_compact = (
    transfers_tab_all
    .groupby(["date", "origin unit"], dropna=False, sort=False)
    .agg(
        **{
            "destination unit": ("destination unit",
                                 lambda s: " ; ".join(s.astype(str).tolist()) if len(s) else None),
            "number of transfer": ("patient (number)", lambda s: int(np.sum(s.values)) if len(s) else None),
            "type of transfer": ("type (new transfer or backlog)",
                                 lambda s: ", ".join(sorted({t.strip() for t in s.dropna().astype(str)}))
                                           if s.notna().any() else None),
        }
    )
    .reset_index()
    .rename(columns={"origin unit": "unit"})
    [["date", "unit", "destination unit", "number of transfer", "type of transfer"]]
)

# Merge into the main tab
occ_tab_all = occ_tab_all.merge(transfers_compact, on=["date","unit"], how="left")


# ---------- Build heatmap matrices (no separate Overflows/Transfers tabs) ----------
units_order = sorted(occ_tab_all["unit"].unique().tolist())
dates_order = sorted(occ_tab_all["date"].astype(str).unique().tolist())

beds_used_matrix = (
    occ_tab_all.pivot(index="unit", columns="date", values="beds_used")
    .reindex(index=units_order, columns=dates_order)
)
beds_matrix = (
    occ_tab_all.pivot(index="unit", columns="date", values="beds")
    .reindex(index=units_order, columns=dates_order)
)
overflow_matrix = (
    occ_tab_all.pivot(index="unit", columns="date", values="Overflow")
    .reindex(index=units_order, columns=dates_order)
    .fillna(0).astype(int)
)
true_matrix = true_matrix.reindex(index=units_order, columns=dates_order)


# ------------- Draw heatmaps and write the 4-tab combined Excel -------------
# Source: Tab 1 "Optimization bedroster" (combined across dates)
# Columns present: date, unit, beds_used, beds, OptimizedBedroster%
assert set(["date","unit","beds_used","beds"]).issubset(set(occ_tab_all.columns)), \
    "Expected columns missing in Optimization bedroster sheet."

# Create pivot tables aligned to true_matrix (same unit/date order)
beds_used_matrix = (occ_tab_all
    .pivot(index="unit", columns="date", values="beds_used")
    .reindex(index=units_order, columns=true_matrix.columns))

beds_matrix = (occ_tab_all
    .pivot(index="unit", columns="date", values="beds")
    .reindex(index=units_order, columns=true_matrix.columns))

# Overflow matrix (Units x Dates)
if ('overflows_tab_all' in globals()) and (not overflows_tab_all.empty):
    overflow_matrix = (overflows_tab_all
        .pivot(index="unit", columns="date", values="overflows number")
        .reindex(index=units_order, columns=true_matrix.columns)
        .fillna(0).astype(int))
else:
    # no overflow rows — make a zeros matrix so labeling still works
    overflow_matrix = pd.DataFrame(0, index=units_order, columns=true_matrix.columns).astype(int)

# If you also want Occ counts for other uses:
occ_count_matrix = (beds_used_matrix.add(overflow_matrix, fill_value=0)).astype(int)


import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

def heatmap_true_with_beds_and_overflow(pct_df, beds_used_mat, beds_mat, overflow_mat,
                                        title, out_png, vmin=0.0, vmax=110.0,
                                        show_zero_overflow=False):
    # Green (0%) → Dark Red (110%+)
    cmap = mpl.colormaps.get_cmap("RdYlGn_r") if hasattr(mpl, "colormaps") else plt.get_cmap("RdYlGn_r")

    fig, ax = plt.subplots(figsize=(1.4*len(pct_df.columns)+3, 0.44*len(pct_df.index)+1.6))
    im = ax.imshow(pct_df.values.astype(float), aspect="auto",
                   cmap=cmap, norm=Normalize(vmin=vmin, vmax=vmax))

    ax.set_yticks(range(len(pct_df.index)));   ax.set_yticklabels(pct_df.index)
    ax.set_xticks(range(len(pct_df.columns))); ax.set_xticklabels(pct_df.columns, rotation=45, ha="right")

    # Annotate each cell with: "<percent>%\n<beds_used>/<beds> +<overflow>\n[95–100%: x]"
    for i in range(len(pct_df.index)):
        for j in range(len(pct_df.columns)):
            if pd.isna(pct_df.iloc[i, j]):
                continue

            valp = float(pct_df.iloc[i, j])
            bu   = beds_used_mat.iloc[i, j] if beds_used_mat is not None else None
            bd   = beds_mat.iloc[i, j]      if beds_mat is not None else None
            ov   = overflow_mat.iloc[i, j]  if overflow_mat is not None else 0

            label = f"{valp:.0f}%"

            # line 2: beds_used / beds (+overflow if >0)
            if pd.notna(bu) and pd.notna(bd):
                bu_i = int(round(bu)); bd_i = int(round(bd))
                label += f"\n{bu_i}/{bd_i}"
                if pd.notna(ov) and (show_zero_overflow or int(ov) > 0):
                    label += f" +{int(ov)}"

                # line 3: compute 95–100% usage locally (no external n95_matrix)
                cap95 = math.floor(0.95 * bd_i)
                n95   = max(0, bu_i - cap95)
                if n95 > 0:
                    label += f"\n[95–100%: {n95}]"

            ax.text(j, i, label, ha="center", va="center",
                    fontsize=8, color=("black" if valp < 80 else "white"))

    ax.set_title(title)
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Optimized bedroster %")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close(fig)


#NEW
# ---------- Draw heatmap (unchanged) ----------
png_true = os.path.join(OUTDIR, f"Heatmap_OptimizedOcc_{SOLVE_DATES[0]}_to_{SOLVE_DATES[-1]}.png")
heatmap_true_with_beds_and_overflow(
    true_matrix.round(1),
    beds_used_matrix,
    beds_matrix,
    overflow_matrix,
    "Optimized bedroster (with beds_used / beds + overflow)",
    png_true,
    vmin=0.0, vmax=110.0,
    show_zero_overflow=False
)

# ---------- Write Excel: ONLY the main tab + heatmap ----------
with pd.ExcelWriter(COMBINED_XLSX, engine="xlsxwriter") as writer:
    # 1) Main tab (now includes destination unit, number of transfer, type of transfer)
    occ_tab_all.to_excel(writer, sheet_name="Optimization bedroster", index=False)

    # 2) Format columns on the main tab
    ws = writer.sheets["Optimization bedroster"]
    cols = list(occ_tab_all.columns)

    # Make "number of transfer" a numeric-looking column in Excel
    if "number of transfer" in cols:
        idx_num = cols.index("number of transfer")  # 0-based index
        fmt_int = writer.book.add_format({"num_format": "0"})
        ws.set_column(idx_num, idx_num, 14, fmt_int)  # width + integer format

    # (optional) keep long text columns readable (no wrapping, wider)
    fmt_nowrap = writer.book.add_format({"text_wrap": False})
    for name, width in [("destination unit", 34), ("type of transfer", 18)]:
        if name in cols:
            i = cols.index(name)
            ws.set_column(i, i, width, fmt_nowrap)

    # 3) Heatmap sheet & image (unchanged)
    ws_hm = writer.book.add_worksheet("Heatmap")
    writer.sheets["Heatmap"] = ws_hm
    ws_hm.write(0, 0, f"Optimized bedroster heatmap — {SOLVE_DATES[0]} to {SOLVE_DATES[-1]}")
    ws_hm.write(1, 0, "Cell label shows: % and occ/beds (+overflow if any; [95–100%: x])")
    ws_hm.insert_image(3, 0, png_true)

print(f"[done] Combined Excel written: {COMBINED_XLSX}")

print(f"[imgs] {png_true}")

# ================= END MULTI-DATE ORCHESTRATOR =================


[done] Combined Excel written: ./outputs\optimized_plan_2025-07-06_to_2025-07-12.xlsx
[imgs] ./outputs\Heatmap_OptimizedOcc_2025-07-06_to_2025-07-12.png


In [6]:
# =========================================================
#Add "Actual_occ" heatmap + a MATCHING optimized heatmap
#   - Reads unit/date order from "Optimization bedroster" (results xlsx)
#   - Reads actuals from ACTUALS_XLSX (unit, date, bedroster_rate)
#   - Produces two PNGs with IDENTICAL renderer & figsize
#   - Adds a new sheet "Actual_occ" with the actual PNG
#   - (OPTIONAL) Rewrites "Heatmap" tab to show BOTH images side-by-side
# =========================================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

RESULTS_XLSX = "./outputs/optimized_plan_2025-07-06_to_2025-07-12.xlsx"  # from Block 3
ACTUALS_XLSX = "./actual_bedroster.xlsx"

def _normalize_dates(s):
    return pd.to_datetime(s, errors="coerce").dt.normalize()

def _load_optimal_for_order(xlsx):
    df = pd.read_excel(xlsx, sheet_name="Optimization bedroster")
    df = df.rename(columns={"unit":"Unit","date":"Date",
                            "OptimizedBedroster%":"OccPct",
                            "Optimized Bedroster%":"OccPct"})
    df["Unit"] = df["Unit"].astype(str).str.strip()
    df["Date"] = _normalize_dates(df["Date"])
    units = sorted(df["Unit"].dropna().unique().tolist())
    dates = sorted(df["Date"].dropna().unique().tolist())
    # Build optimized % matrix from the same sheet -> this ensures we can re-render optimized with the same renderer
    df["OccPct"] = pd.to_numeric(df["OccPct"], errors="coerce")
    P_opt = df.pivot_table(index="Unit", columns="Date", values="OccPct", aggfunc="mean").reindex(index=units, columns=dates)
    return units, dates, P_opt

def _load_actuals_long(xlsx):
    df = pd.read_excel(xlsx, sheet_name=0)
    df = df.rename(columns={"unit":"Unit","date":"Date","bedroster_rate":"OccPct"})
    if not {"Unit","Date","OccPct"}.issubset(df.columns):
        raise ValueError("actual_bedroster.xlsx must have columns: unit, date, bedroster_rate")
    df["Unit"]   = df["Unit"].astype(str).str.strip()
    df["Date"]   = _normalize_dates(df["Date"])
    df["OccPct"] = pd.to_numeric(df["OccPct"], errors="coerce")
    s = df["OccPct"].dropna()
    if len(s) and (s.between(0,1).mean() > 0.9) and (s.max() <= 1.1):
        df["OccPct"] = df["OccPct"] * 100.0
    return df[["Unit","Date","OccPct"]].dropna(subset=["Unit","Date"])

def _pivot(df_long, units, dates):
    P = df_long.pivot_table(index="Unit", columns="Date", values="OccPct", aggfunc="mean")
    return P.reindex(index=units, columns=dates)

from matplotlib.colors import Normalize

def _save_heatmap(P, title, out_png, figsize, annotate=True, precision=0, brown_over_100=False):
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import matplotlib as mpl
    from matplotlib.colors import Normalize, BoundaryNorm

    # Colormap
    cmap = (mpl.colormaps.get_cmap("RdYlGn_r").copy()
            if hasattr(mpl, "colormaps") else plt.get_cmap("RdYlGn_r").copy())

    if brown_over_100:
        # map 0–100 to RdYlGn_r, anything >100 to brown
        cmap.set_over("#8B4513")      # SaddleBrown
        norm = Normalize(vmin=0.0, vmax=100.0)
        cbar_kwargs = {"extend": "max"}
    else:
        norm = Normalize(vmin=0.0, vmax=110.0)
        cbar_kwargs = {}

    fig, ax = plt.subplots(figsize=figsize, dpi=220)
    im = ax.imshow(P.values.astype(float), aspect="auto", cmap=cmap, norm=norm)

    ax.set_yticks(range(len(P.index)));   ax.set_yticklabels(P.index)
    ax.set_xticks(range(len(P.columns))); ax.set_xticklabels(
        [pd.Timestamp(d).strftime("%Y-%m-%d") for d in P.columns], rotation=45, ha="right"
    )
    ax.set_title(title)
    ax.set_xticks(np.arange(-.5, P.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, P.shape[0], 1), minor=True)
    ax.grid(which="minor", linewidth=0.2)

    # BLACK annotations
    if annotate:
        for i in range(P.shape[0]):
            for j in range(P.shape[1]):
                val = P.iat[i, j]
                if pd.notna(val):
                    ax.text(j, i, f"{val:.{precision}f}%", ha="center", va="center",
                            fontsize=8, color="black", fontweight="normal")

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, **cbar_kwargs)
    cbar.set_label("Bedroster %")

    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


# 1) Read order + optimized matrix from results
units, dates, P_opt = _load_optimal_for_order(RESULTS_XLSX)
# 2) Build actual matrix on the exact same grid
actual_long = _load_actuals_long(ACTUALS_XLSX)
P_act = _pivot(actual_long[actual_long["Unit"].isin(units) & actual_long["Date"].isin(dates)], units, dates)

# Shared geometry so the two PNGs match exactly
shared_figsize = (max(6, 0.45*len(dates)), max(4, 0.35*len(units)))
start_s = str(pd.Timestamp(dates[0]).date()) if dates else "NA"
end_s   = str(pd.Timestamp(dates[-1]).date()) if dates else "NA"
out_dir = os.path.dirname(RESULTS_XLSX) or "."

png_opt = os.path.join(out_dir, f"occ_optimized_{start_s}_to_{end_s}.png")
png_act = os.path.join(out_dir, f"occ_actual_{start_s}_to_{end_s}.png")

# Re-render optimized with the shared renderer (so both PNGs are identical in style/size)
_save_heatmap(P_opt.round(1), "Optimized bedroster (%)", png_opt, figsize=shared_figsize)
_save_heatmap(P_act.round(1), "Actual bedroster (%)",    png_act, figsize=shared_figsize)

# 3) Add a new sheet with the Actual PNG (non-destructive save)
new_xlsx = os.path.join(out_dir, f"optimized_plan_actualcomp_{start_s}_to_{end_s}.xlsx")
wb = load_workbook(RESULTS_XLSX)
sheet_name = "Actual_occ"
while sheet_name in wb.sheetnames:
    sheet_name += "_1"
ws = wb.create_sheet(sheet_name)
ws["A1"] = "Actual bedroster (%)"; ws["A2"] = f"Window: {start_s} to {end_s}"; ws["A3"] = f"Generated: {datetime.now():%Y-%m-%d %H:%M}"

img = XLImage(png_act); ws.add_image(img, "A6")
wb.save(new_xlsx)
print(f"[done] Wrote: {os.path.abspath(new_xlsx)}")

#Also place both images on the original 'Heatmap' tab side-by-side:
# =========================================================
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

# Reopen the newly saved file
wb = load_workbook(new_xlsx)
sheet_compare = "Heatmap_compare"
if sheet_compare in wb.sheetnames:
    del wb[sheet_compare]
ws_cmp = wb.create_sheet(sheet_compare)

ws_cmp["A1"] = f"Optimized vs Actual bedroster comparison ({start_s} to {end_s})"
ws_cmp["A2"] = "Both images are generated with identical size, color scale (0–110%), and layout for direct visual comparison."
ws_cmp["A3"] = f"Generated: {datetime.now():%Y-%m-%d %H:%M}"

# Insert the optimized and actual heatmaps side by side
img_opt = XLImage(png_opt)
img_act = XLImage(png_act)

# Place optimized on the left, actual on the right
ws_cmp.add_image(img_opt, "A6")
ws_cmp.add_image(img_act, "O6")   # shift horizontally so they don't overlap

wb.save(new_xlsx)
print(f"[compare] Added side-by-side comparison sheet: {sheet_compare}")
print(f"[done] Final workbook: {os.path.abspath(new_xlsx)}")

# --- Add ActualOcc% next to Overflow in "Optimization bedroster" (first tab) ---

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
import matplotlib.pyplot as plt


# --------- CONFIG ---------
RESULTS_XLSX = "./outputs/optimized_plan_actualcomp_2025-07-06_to_2025-07-12.xlsx"  # <-- set yours if different
ACTUALS_XLSX = "./inputs/actual_bedroster.xlsx"  # or just "actual_bedroster.xlsx" if in project root
OPT_SHEET = "Optimization bedroster"
NEW_COL_NAME = "ActualOcc%"   # new column to add right after "Overflow"

# Shared color scale (fixed 0–100 recommended for bedroster)
vmin, vmax = 0, 100
FIGSIZE = (12, 8); DPI = 220

def plot_heatmap(matrix, title, outfile):
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    im = ax.imshow(matrix.to_numpy(dtype=float), aspect="auto", interpolation="nearest",
                   vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(matrix.shape[1]))
    ax.set_xticklabels([str(d) for d in matrix.columns], rotation=45, ha="right")
    ax.set_yticks(np.arange(matrix.shape[0]))
    ax.set_yticklabels([str(u) for u in matrix.index])
    ax.set_title(title)
    ax.set_xticks(np.arange(-.5, matrix.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, matrix.shape[0], 1), minor=True)
    ax.grid(which="minor", linewidth=0.2)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Bedroster %")
    fig.tight_layout()
    fig.savefig(outfile, bbox_inches="tight")
    plt.close(fig)

# --------- HELPERS ---------
def _normalize_date_col(s):
    s = pd.to_datetime(s, errors="coerce")
    # Normalize to date (no time component) to ensure exact match with model sheet
    return s.dt.normalize()

def load_actuals(path):
    # Expect columns: unit, date, bedroster_rate (0–1 or 0–100)
    df = pd.read_excel(path, sheet_name=0)
    df = df.rename(columns={
        "unit": "Unit",
        "date": "Date",
        "bedroster_rate": "OccPct"
    })
    needed = {"Unit","Date","OccPct"}
    if not needed.issubset(df.columns):
        raise ValueError(f"actual_bedroster.xlsx must contain columns {needed}. Found: {list(df.columns)}")
    # Clean and coerce
    df["Unit"] = df["Unit"].astype(str).str.strip()
    df["Date"] = _normalize_date_col(df["Date"])
    df["OccPct"] = pd.to_numeric(df["OccPct"], errors="coerce")

    # If values look like 0–1, scale to 0–100
    s = df["OccPct"].dropna()
    if len(s) and (s.between(0,1).mean() > 0.9) and (s.max() <= 1.1):
        df["OccPct"] = df["OccPct"] * 100.0

    # Round for neatness; keep one decimal in case you care about halves
    df["OccPct"] = df["OccPct"].round(1)
    return df[["Unit","Date","OccPct"]]

def load_opt_sheet(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet)
    # Make sure required columns exist
    colmap = {c.lower(): c for c in df.columns}
    for req in ["unit","date","overflow"]:
        if req not in colmap:
            raise ValueError(f"'{sheet}' must have a '{req.capitalize() if req!='overflow' else 'Overflow'}' column. Found: {list(df.columns)}")
    # Normalize key columns
    df = df.rename(columns={
        colmap["unit"]: "Unit",
        colmap["date"]: "Date"
    })
    df["Unit"] = df["Unit"].astype(str).str.strip()
    df["Date"] = _normalize_date_col(df["Date"])
    return df

def derive_outfile(path):
    # Insert/ensure suffix "_actualcomp_" before the date range if present; else add suffix
    p = Path(path)
    stem = p.stem
    if "_actualcomp_" in stem:
        return str(p)  # already has suffix
    # Try to find a date span like 2025-07-06_to_2025-07-12
    m = re.search(r"(\d{4}-\d{2}-\d{2}_to_\d{4}-\d{2}-\d{2})", stem)
    if m:
        new_stem = stem.replace(m.group(1), f"actualcomp_{m.group(1)}")
    else:
        new_stem = stem + "_actualcomp"
    return str(p.with_stem(new_stem))

# --------- LOAD ---------
if not Path(RESULTS_XLSX).exists():
    raise FileNotFoundError(f"Results file not found: {RESULTS_XLSX}")
if not Path(ACTUALS_XLSX).exists():
    # also try project root if not in inputs
    if Path("./actual_bedroster.xlsx").exists():
        ACTUALS_XLSX = "./actual_bedroster.xlsx"
    else:
        raise FileNotFoundError(f"Actuals file not found at {ACTUALS_XLSX} or ./actual_bedroster.xlsx")

opt_df = load_opt_sheet(RESULTS_XLSX, OPT_SHEET)
act_df = load_actuals(ACTUALS_XLSX)

# --------- MATCH & MERGE (Unit+Date) ---------
merged = opt_df.merge(act_df, on=["Unit","Date"], how="left", suffixes=("", "_ACT"))
# OccPct holds the actual bedroster % from the actuals file
merged[NEW_COL_NAME] = merged["OccPct"].round(1)
merged.drop(columns=["OccPct"], inplace=True)

# --------- WRITE INTO THE EXISTING SHEET, NEXT TO "Overflow" ---------
wb = load_workbook(RESULTS_XLSX)
ws = wb[OPT_SHEET]

# Find header row (assume first row is headers) and locate "Overflow" column index
header_row = 1
headers = [cell.value if cell.value is not None else "" for cell in ws[header_row]]
try:
    overflow_idx = headers.index("Overflow") + 1  # 1-based index in openpyxl
except ValueError:
    # case-insensitive fallback
    idxs = [i for i,h in enumerate(headers, start=1) if str(h).strip().lower()=="overflow"]
    if not idxs:
        raise ValueError("Could not find 'Overflow' header in the sheet.")
    overflow_idx = idxs[0]

# Determine where to write the new column (right after Overflow)
write_col = overflow_idx + 1

# If a column already exists there, insert a new one so we don't overwrite
ws.insert_cols(write_col)

# Write header
ws.cell(row=header_row, column=write_col, value=NEW_COL_NAME)

# Build a quick lookup (Unit, Date) -> ActualOcc%
# We'll align by reading Unit/Date values present in the sheet
unit_col_idx = headers.index("Unit") + 1 if "Unit" in headers else [i for i,h in enumerate(headers,1) if str(h).lower()=="unit"][0]
date_col_idx = headers.index("Date") + 1 if "Date" in headers else [i for i,h in enumerate(headers,1) if str(h).lower()=="date"][0]

# Create a dict for fast lookup
key_to_actual = {(u, pd.to_datetime(d).normalize() if pd.notna(d) else pd.NaT): v
                 for u, d, v in merged[["Unit","Date", NEW_COL_NAME]].itertuples(index=False, name=None)}

# Write values row by row
for r in range(header_row+1, ws.max_row+1):
    u = ws.cell(row=r, column=unit_col_idx).value
    d = ws.cell(row=r, column=date_col_idx).value
    d_norm = pd.to_datetime(d).normalize() if d is not None else pd.NaT
    val = key_to_actual.get((str(u).strip() if u is not None else "", d_norm), None)
    ws.cell(row=r, column=write_col, value=float(val) if val is not None and pd.notna(val) else None)

# Save to a new file with _actualcomp_ suffix (non-destructive)
OUTFILE = derive_outfile(RESULTS_XLSX)
wb.save(OUTFILE)
print(f"[done] Wrote {NEW_COL_NAME} next to 'Overflow' in '{OPT_SHEET}'.")
print(f"[out] {OUTFILE}")


[done] Wrote: C:\Users\olk716\ILP model one day append\outputs\optimized_plan_actualcomp_2025-07-06_to_2025-07-12.xlsx
[compare] Added side-by-side comparison sheet: Heatmap_compare
[done] Final workbook: C:\Users\olk716\ILP model one day append\outputs\optimized_plan_actualcomp_2025-07-06_to_2025-07-12.xlsx
[done] Wrote ActualOcc% next to 'Overflow' in 'Optimization bedroster'.
[out] outputs\optimized_plan_actualcomp_2025-07-06_to_2025-07-12.xlsx
